In [0]:
BRONZE_PATH = "/Volumes/workspace/default/insure_data/bronze"
SILVER_PATH = "/Volumes/workspace/default/insure_data/silver"

In [0]:
from pyspark.sql import functions as F

agent_df = spark.read.format("delta").load(f"{BRONZE_PATH}/agent")

silver_agent_df = (
    agent_df

    # Remove duplicate agents
    .dropDuplicates(["agent_no"])

    # Standardize text
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("middle_name", F.initcap(F.trim(F.col("middle_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("branch_name", F.initcap(F.trim(F.col("branch_name"))))
    .withColumn("agent_status", F.upper(F.trim(F.col("agent_status"))))

    # Derived Columns
    .withColumn(
        "agent_full_name",
        F.concat_ws(
            " ",
            "first_name",
            "middle_name",
            "last_name"
        )
    )

    .withColumn(
        "agent_location",
        F.concat_ws(
            ", ",
            "branch_name",
            "city"
        )
    )
)

(
    silver_agent_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{SILVER_PATH}/agent")
)

In [0]:
print("Record Count:",
      silver_agent_df.count())

silver_agent_df.printSchema()

In [0]:
# 2 Cuastomer
from pyspark.sql import functions as F

customer_df = spark.read.format("delta").load(f"{BRONZE_PATH}/customer")

silver_customer_df = (
    customer_df

    .dropDuplicates(["customer_no"])

    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("customer_status", F.upper(F.trim(F.col("customer_status"))))
    .withColumn("customer_type", F.upper(F.trim(F.col("customer_type"))))

    .withColumn(
        "cust_full_name",
        F.concat_ws(
            " ",
            "first_name",
            "last_name"
        )
    )
)

(
    silver_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/customer")
)

In [0]:
print("Record Count:",
      silver_customer_df.count())

silver_customer_df.printSchema()

In [0]:
# 3 Policy
from pyspark.sql import functions as F

policy_df = spark.read.format("delta").load(f"{BRONZE_PATH}/policy")

silver_policy_df = (
    policy_df

    .dropDuplicates(["policy_no"])

    .withColumn(
        "product_id",
        F.upper(F.trim(F.col("product_id")))
    )

    .withColumn(
        "policy_status",
        F.upper(F.trim(F.col("policy_status")))
    )
)

(
    silver_policy_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/policy")
)

In [0]:
print("Record Count:",
      silver_policy_df.count())

silver_policy_df.printSchema()

In [0]:
# read column names
spark.read.format("delta").load(f"{BRONZE_PATH}/agent_policy").printSchema()

spark.read.format("delta").load(f"{BRONZE_PATH}/customer_policy").printSchema()

spark.read.format("delta").load(f"{BRONZE_PATH}/money_in_dtl").printSchema()

spark.read.format("delta").load(f"{BRONZE_PATH}/product_master").printSchema()

spark.read.format("delta").load(f"{BRONZE_PATH}/product_commission_rule").printSchema()

spark.read.format("delta").load(f"{BRONZE_PATH}/customer_role_master").printSchema()

In [0]:
# 4 Pol_agent map

from pyspark.sql import functions as F

agent_policy_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/agent_policy")
)

silver_agent_policy_df = (

    agent_policy_df

    .dropDuplicates(["policy_no","agent_no"])

    .withColumn(
        "split_percentage",
        F.round(F.col("split_percentage"),2)
    )

)

(
    silver_agent_policy_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/agent_policy")
)

In [0]:
# 5 Pol_cust map
from pyspark.sql import functions as F

customer_policy_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/customer_policy")
)

silver_customer_policy_df = (

    customer_policy_df

    .dropDuplicates(["policy_no","customer_no"])

    .withColumn(
        "relationship_type",
        F.upper(F.trim(F.col("relationship_type")))
    )

)

(
    silver_customer_policy_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/customer_policy")
)

In [0]:
#  6 Money In
from pyspark.sql import functions as F

money_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/money_in_dtl")
)

silver_money_df = (

    money_df

    .dropDuplicates(["payment_id"])

    .withColumn(
        "payment_type",
        F.upper(F.trim(F.col("payment_type")))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )

    .filter(F.col("premium_amount") > 0)

)

(
    silver_money_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/money_in_dtl")
)

In [0]:
# 7 Product master
from pyspark.sql import functions as F

product_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/product_master")
)

silver_product_df = (

    product_df

    .dropDuplicates(["product_id"])

    .withColumn(
        "product_name",
        F.initcap(F.trim(F.col("product_name")))
    )

    .withColumn(
        "category",
        F.upper(F.trim(F.col("category")))
    )

)

(
    silver_product_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/product_master")
)

In [0]:
# 8 commission rule
from pyspark.sql import functions as F

rule_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/product_commission_rule")
)

silver_rule_df = (

    rule_df

    .dropDuplicates(["rule_id"])

)

(
    silver_rule_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/product_commission_rule")
)

In [0]:
# 9 customer role table
from pyspark.sql import functions as F

role_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/customer_role_master")
)

silver_role_df = (

    role_df

    .dropDuplicates(["customer_no"])

    .withColumn(
        "role_type",
        F.upper(F.trim(F.col("role_type")))
    )

)

(
    silver_role_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{SILVER_PATH}/customer_role_master")
)

In [0]:
# check count
print(f"agent_policy : {silver_agent_policy_df.count()}")

print(f"customer_policy : {customer_policy_df.count()}")

print(f"money_in_dtl : {silver_money_df.count()}")

print(f"product_master : {silver_product_df.count()}")

print(f"product_commission_rule : {silver_rule_df.count()}")

print(f"customer_role_master : {silver_role_df.count()}")